<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **7. Introduction to Local LLMs**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 7.1 Introduction

[05.05_Introduction_to_LLMs.ipynb](05.05_Introduction_to_LLMs.ipynb) showed how large language models work. The largest of them need clusters of accelerators and are usually offered as cloud services. Smaller models make it possible to run an LLM on your own device instead. Such a **local LLM** keeps your data with you, works without a network, and costs nothing per request, but it is limited by the device it runs on.

In this notebook we look at the local models on your Raspberry Pi, at the Swiss models among them and how they got onto the Pi, and at what running a model locally means for an organisation. [05.08_Hardware_for_Local_LLMs.ipynb](05.08_Hardware_for_Local_LLMs.ipynb) then looks at the hardware they run on. Allow about **60 minutes**, plus time for the questions.

By the end, you should be able to say which models are on your Pi and what each one is for, estimate how large a model file is from its parameters, explain how a model gets from Hugging Face onto the Pi, and name the responsibilities that come with running a model locally.

#### 7.1.1 Environment Setup

Everything that involves a model runs on the Pi, so connect it as in [05.04_GettingStarted_with_Raspberry_Pi.ipynb](05.04_GettingStarted_with_Raspberry_Pi.ipynb).

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

### 7.2 The Models on Your Pi

The Pi carries one to three models for each topic of this course. Each is small enough for 8 GB of memory, and they differ enough to be worth comparing.

| Model | Made by | What it is for |
|:------|:--------|:---------------|
| `qwen3.5:0.8b`, `qwen3.5:4b`, `qwen3.5:9b` | Alibaba Qwen | one family in three sizes, so you can compare sizes. `4b` is the everyday default |
| `granite4.2:3b` | IBM | a small reasoning model whose thinking can be switched on and off |
| `lfm2.5` | Liquid AI | agents and tool calls. Only a fraction of its weights is used for each token |
| `granite4:1b-h` | IBM | a hybrid of Mamba-2 and Transformer layers with a context window of a million tokens |
| `gemma4:e2b-it-qat` | Google DeepMind | built for edge devices and trained to survive 4-bit quantization |
| `qwen2.5-coder:7b`, `qwen2.5-coder:1.5b-base` | Alibaba Qwen | code: the 7B writes it when asked, the 1.5B only continues it, for autocomplete in VS Code ([05.17](05.17_Local_Agents_II.ipynb)) |
| `apertus-v1.1:1.5b`, `apertus-v1.1:4b`, `apertus-v1.5:8b` | ETH Zurich, EPFL and CSCS | the Swiss open models, Section 7.4 |
| `tev1:0.8b`, `tev1:4b` | Together AI | decision models: they choose among answers you define and return probabilities instead of text ([05.14](05.14_New_Kinds_of_Language_Models.ipynb)) |
| `nomic-embed-text-v2-moe`, `all-minilm` | Nomic AI, sentence-transformers | embeddings for search and RAG, not for chat |

#### 7.2.1 How big is a model?

A model file is mostly its weights: the number of parameters times the bits each one is stored with. Most models here use 4-bit quantization (`Q4_K_M`, about 4.8 bits per weight once its scaling factors are counted), a few use 8 bits (`Q8_0`, 8.5 bits), and the small embedding models 16 bits (`F16`). [05.09_Making_Models_Smaller.ipynb](05.09_Making_Models_Smaller.ipynb) explains these formats.

So you can predict each file's size from two numbers that `caspi.models(info=True)` reports. The cell compares that prediction with the real size of every model on your Pi:

In [ ]:
import caspi

caspi.connect()

BITS = {"Q4_K_M": 4.8, "Q4_0": 4.5, "Q8_0": 8.5, "F16": 16, "BF16": 16}


def billions(parameters):
    """'3.7B' -> 3.7, '752.39M' -> 0.75"""
    number, unit = float(parameters[:-1]), parameters[-1]
    return number if unit == "B" else number / 1000


for model in caspi.models(info=True):
    bits = BITS.get(model["quantization"])
    expected = f"{billions(model['parameters']) * bits / 8:5.2f}" if bits else "    ?"
    print(f"{model['name']:30s} {model['parameters']:>8s} in {model['quantization']:7s}"
          f" expected {expected} GB, real {model['size_gb']:5.2f} GB")

Most predictions land within a few percent: a model file is little more than its weights. The exceptions are more interesting. `qwen3.5:0.8b` and `tev1:0.8b` report the same number of parameters in the same format, yet one file is half a gigabyte larger. `tev1` is a version of `qwen3.5` trained to read text only. `qwen3.5` also carries a vision encoder, the part that turns an image into tokens, and the parameter count leaves it out. That is why all three `qwen3.5` files are larger than predicted. Section 7.5.1 puts the encoder to use. `gemma4:e2b-it-qat` is off by even more: half of its 4.6 billion parameters sit in one large table, an embedding for every layer, stored with about 6.6 bits per weight instead of the 4.5 of the rest. The model looks up only a few rows of that table for each token, which is why Google calls it an "effective" 2B model.

`caspi.show()` tells you more about one model. Compare a dense model with `lfm2.5`, a mixture of experts:

In [ ]:
caspi.show("granite4.2:3b")
caspi.show("lfm2.5")

The full model description holds even more. For `lfm2.5` it says how many experts each layer has and how many of them are used for each token:

In [ ]:
info = caspi.ollama("/api/show", {"model": "lfm2.5"})["model_info"]
print("experts per layer :", info["lfm2moe.expert_count"])
print("used per token    :", info["lfm2moe.expert_used_count"])

Each of these models was made small on purpose, and four techniques do nearly all of that work: **quantization** (storing each weight with fewer bits), **pruning** (removing the weights that contribute least), **distillation** (training a small model to imitate a large one) and **selective parameter activation** (keeping all weights but using only a few of them per token). [05.09](05.09_Making_Models_Smaller.ipynb) applies them to a real model, measures what each one costs, and lets you find out which of them made the models on your Pi small.

### 7.3 Running a Model Locally: Who Is Responsible?

When an organisation runs a language model on its own hardware, the main advantage is that sensitive data no longer has to leave its infrastructure. The responsibility moves inside with it, and some risks remain:

- **Bias and fairness:** a local model can reflect stereotypes from its training data. If its output influences decisions about customers, staff or partners, the organisation is accountable for them, not the model's vendor.
- **Transparency:** if a model's output affects people or business outcomes, someone must be able to explain how it was produced. In regulated industries, missing auditability is a legal risk.
- **Security and maintenance:** a cloud provider patches and monitors its service. A local model has to be patched, monitored and access-controlled by the organisation itself.
- **Capacity:** running and fine-tuning models takes expertise and hardware. A smaller, specialised model is often the better balance of capability, cost and risk.

Your own Pi shows how quickly this matters. Its Ollama answers anyone who can reach 172.31.250.1 on port 11434, without a password. On a course Pi that is fine. In a company, someone has to notice such a gap, close it, watch what the model is used for, and decide what it may be used for.

### 7.4 Swiss AI: Apertus

**Apertus** is the language model of the Swiss AI Initiative, built by ETH Zurich, EPFL and CSCS, the Swiss National Supercomputing Centre, and trained on the Alps supercomputer in Lugano [01]. Its name is Latin for *open*, and openness is what sets it apart. Most models that call themselves open publish their weights, the finished numbers. Apertus also publishes its training data and the code and recipes that produced it, so anyone can check what went in and could train it again. It learned from text in 1811 languages, about 40 % of it not in English, including Swiss German and Romansh. Its builders also left out web pages whose owners had asked not to be used for AI training, even pages collected before the owners asked.

Your Pi carries three versions:

| On your Pi | Version | Parameters | File | What is new |
|:-----------|:--------|:-----------|:-----|:------------|
| `apertus-v1.1:1.5b` | 1.1, May 2026 | 1.5 billion | 1.6 GB, `Q8_0` | small Apertus models for phones and laptops, distilled from the 8B model [02] |
| `apertus-v1.1:4b` | 1.1, May 2026 | 3.8 billion | 2.4 GB, `Q4_K_M` | the same, larger |
| `apertus-v1.5:8b` | 1.5, July 2026 | 8.1 billion | 5.1 GB, `Q4_K_M` | four trillion more tokens of text, images and audio, a thinking mode, and a context of 262,144 tokens [03] |

The first Apertus, 1.0 from September 2025, came in two sizes: 8 and 70 billion parameters. The 70B model needs more than 40 GB even with 4-bit weights and fits on no Pi. Version 1.1, published as Apertus Mini, added three small models (0.5, 1.5 and 4 billion parameters) and made them with distillation, the technique of [05.09](05.09_Making_Models_Smaller.ipynb), Section 9.6: the 8B model was the teacher, and the small ones learned to reproduce its predictions. Version 1.5 trained both sizes further, now also on images and audio; only the 8B fits on the Pi. Its thinking mode is not available on the Pi: Ollama does not offer it for this model, and `think=True` fails with the message that the model does not support thinking.

#### 7.4.1 From Hugging Face to your Pi

Ollama's official library has no Apertus, only uploads by individual users, so we built the three models ourselves from the official weights on [Hugging Face](https://huggingface.co/swiss-ai). A short script on a laptop downloads each model, converts it with [llama.cpp](https://github.com/ggml-org/llama.cpp), the engine inside Ollama, into a single GGUF file, quantizes it (05.09) and makes sure the model stops at the end of its turn. For Apertus 1.5 it first removes the image and audio parts, which is why the version on your Pi reads text only.

The next cell shows what arrived on the Pi. `caspi.show()` reads it from Ollama, as in Section 7.2.

In [ ]:
for model in ("apertus-v1.1:1.5b", "apertus-v1.1:4b", "apertus-v1.5:8b"):
    caspi.show(model)

The architecture is `apertus`. The context is 4096 tokens for version 1.1 and 262,144 for version 1.5, of which the Pi uses 8192. The defaults are the settings we gave each model when we built it: the context, and temperature 0.8 and top-p 0.9, which the model cards of versions 1.0 and 1.1 recommend. The card of version 1.5 recommends no values, so we kept them.

The chat template also came along from Hugging Face. It is the format that marks who is speaking ([05.13](05.13_Experimenting_with_Prompts.ipynb), Section 13.2), written in Jinja, the template language of Hugging Face, and Ollama's llama.cpp engine runs it as it is:

In [ ]:
template = caspi.ollama("/api/show", {"model": "apertus-v1.1:4b"})["template"]
print(template[:2000].replace("{%", "\n{%"))   # its beginning, one Jinja tag per line

<div class="alert alert-block alert-info">

Find the tokens that open and close the user's turn and the assistant's turn. When the model writes the token that closes its turn, Ollama ends the answer. What would happen if the converted file named the wrong token (05.13, Section 13.5.1)? What do the lines `Deliberation: disabled` and `Tool Capabilities: disabled` switch off?
</div>

#### 7.4.2 Swiss German, Romansh and English

Apertus was trained with far more non-English text than most models. Does that show on the Pi? The next cells ask each model the same three questions, one in English, one in Zurich German and one that asks for an answer in Romansh, and print the answers with the speed. For comparison, the last cell asks `qwen3.5:4b`, a model of similar size from Alibaba.

<div class="alert alert-block alert-warning">

**Free the memory first.** `apertus-v1.5:8b` takes more than 6 GB once it is loaded, three quarters of the Pi. Next to any other model it can freeze the Pi (05.04, Section 4.5.2). `compare` therefore runs `caspi.stop()` before every model and frees the memory again at the end. Do the same whenever you load a large model yourself.
</div>

<div class="alert alert-block alert-info">

Predict before you run: which model answers best in Swiss German, and which one in Romansh? Which one is fastest? Together the cells take more than ten minutes, most of it for loading: in our run, the four models took nine minutes to load.
</div>

In [ ]:
QUESTIONS = {
    "English": "In two sentences: what is the Röstigraben?",
    "Zurich German": "Erklär mer i zwei Sätz uf Züritüütsch, was de Röstigrabe isch.",
    "Romansh": "Answer in Romansh, in one sentence: which languages are spoken in Switzerland?",
}


def compare(model, **settings):
    """Load one model alone, ask every question, and free the memory again."""
    caspi.stop()
    caspi.load(model)
    for language, question in QUESTIONS.items():
        reply = caspi.chat(model, question, temperature=0, num_predict=150, show=False, **settings)
        print(f"--- {language}, {reply.stats['reply_tokens_per_s']:.1f} tokens per second\n{reply.text}\n")
    caspi.stop(model)


compare("apertus-v1.1:1.5b")

In [ ]:
compare("apertus-v1.1:4b")

In [ ]:
compare("apertus-v1.5:8b")

In [ ]:
compare("qwen3.5:4b", think=False)

In our run, all three Apertus models knew in English that the Röstigraben is the line between German- and French-speaking Switzerland, but two of them added a wrong detail: the 1.5B model placed it "along the Swiss-German border", and the 8B model called it a divide between the north and the south of the country, although French is spoken in the west. In Zurich German, only the 1.5B model stayed close to the truth before `num_predict=150` cut its answer off. The 4B model described a region of forests and mountains in the middle of Switzerland, and the 8B model a dish of potatoes, onions, meat and cheese. Both are hallucinations, as in 05.13: fluent, confident and wrong. In Romansh, the 4B and 8B models listed the four national languages correctly, while the 1.5B model wrote fluent Romansh that called Romansh the main language of Switzerland and German the official one.

`qwen3.5:4b` got the first two wrong. In English it made the Röstigraben a region of four cantons named after a dish, and in Zurich German a border between East and West Germany that lasted until October 2024. In Romansh it named the four languages, but with words such as "frantsch" and "lingua germana" where the Apertus models wrote "franzos" and "tudestg", the Romansh words.

The speeds follow the file sizes, as in [05.08](05.08_Hardware_for_Local_LLMs.ipynb): about 9 tokens per second for the 1.5B model, 5 for the 4B and 2 for the 8B. `qwen3.5:4b` managed 3.5, slower than the Apertus 4B because its file is larger, 3.3 GB against 2.4. File size times speed comes out at 11 to 15 GB per second for all four, below the 17 GB per second that the Pi's memory can deliver at most (05.08, Section 8.4.2).

<div class="alert alert-block alert-info">

You are the judge for Zurich German. For Romansh, ask someone who speaks it, or look up a few words in the dictionary [Pledari Grond](https://www.pledarigrond.ch). Rate each answer on two things separately: does it sound like the language, and is it true? Then add a question of your own about Switzerland to `QUESTIONS`, in any of the four national languages, and run `compare` again with the model you found best.
</div>

### 7.5 Building on Local Models

A small model becomes much more useful when it is combined with other parts. The rest of this chapter builds several of them:

- **Retrieval-augmented generation (RAG):** the model looks things up in your own documents before it answers, so it can stay small and still be specific ([05.10](05.10_Local_RAG_meets_Tit-for-Tat.ipynb)).
- **Agents:** the model plans, calls tools and works in a loop toward a goal ([05.15](05.15_Local_Agents_I.ipynb), [05.17](05.17_Local_Agents_II.ipynb)).
- **Tools and the Model Context Protocol:** a standard way to connect a model to your applications and data ([05.16](05.16_Local_Model_Context_Protocol.ipynb)).
- **Several models together:** small specialised models that check, vote on or refine each other's answers.
- **Images:** `qwen3.5` and `gemma4` also accept images, so a local model can answer questions about a photo without sending it anywhere. Apertus 1.5 can too, but not in the version on your Pi (Section 7.4.1).

#### 7.5.1 Try it: a model that sees

`qwen3.5:4b` accepts images, and the picture never travels further than the cable to your Pi. The model turns an image into a few hundred tokens, and on the Pi every token costs time, so we shrink the photo of the Pi from 05.04 to 512 pixels first. If the model is not in memory yet, loading it takes about two and a half minutes.

In [ ]:
import base64
import io

from PIL import Image, ImageOps


def photo(path, size=512):
    """An image file as the model receives it: scaled down, as JPEG, encoded as text."""
    image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")   # turn phone photos upright
    image.thumbnail((size, size))
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode()


caspi.ask("qwen3.5:4b", "What is in this picture? Answer in two sentences.",
          images=[photo("../../.admin/notebooks/picture/raspberrypi.png")], think=False)

Check the answer against the photo in [05.04](05.04_GettingStarted_with_Raspberry_Pi.ipynb), where the board says "Raspberry Pi 5" in white letters. When we tried it on a course Pi, the model called it a Raspberry Pi 4, with complete confidence. A small model can sound sure of itself and still be wrong, and with images that is harder to notice than with text.

<div class="alert alert-block alert-info">

Take a photo of something on your desk, save it as a JPEG or PNG file next to this notebook, and put its file name into `MY_PHOTO`. iPhones save photos as HEIC, which the code cannot open; export a copy as JPEG. If you give a full path on Windows, write it as `r"C:\..."`. Until you do, the cell uses a picture of the Raspberry Pi AI Kit, an add-on board with an accelerator for AI models. What does it get right, and what does it miss? Then try `size=1024`: the timing line shows what the larger image costs in prompt tokens and seconds.
</div>

In [ ]:
MY_PHOTO = "../../.admin/notebooks/picture/raspberrypi_AI_kit.png"   # your photo here

caspi.ask("qwen3.5:4b", "What is in this picture? Answer in two sentences.",
          images=[photo(MY_PHOTO, size=512)], think=False)

### 7.6 Summarizing Remarks

A local model is a file, and its size follows from its parameters and the bits per weight, plus whatever the parameter count leaves out, such as a vision encoder. The models on your Pi differ in what they were built for, and choosing one is a trade between what it knows, how fast it answers and how much memory it needs. 05.08 looks at the hardware behind that speed, and 05.09 at how the models were made small.

Apertus shows what open can mean: weights, data and recipes, all public, from Swiss universities. It also shows the work between a published model and a running one. Every step of the conversion, from the format to the quantization to the end token, decides how the model behaves on your Pi, and none of it is visible in the answers.

Running a model locally keeps the data with you, and it moves the responsibility to you: for what the model says, for who can reach it, and for what it is used for.

### 7.7 Further Reading and Literature

[01] Project Apertus (2025). **Apertus: Democratizing Open and Compliant LLMs for Global Language Environments.** Technical report. [Paper](https://arxiv.org/abs/2509.14233)

[02] Panferov, A., Melikidze, D., Jaggi, M., & Alistarh, D. (2026). **Apertus LLM Family Expansion via Distillation and Quantization.** [Paper](https://arxiv.org/abs/2605.29128). How the small models of version 1.1 were made.

[03] Project Apertus (2026). **Apertus 1.5: From Open Text Models to Open Multimodal Reasoners.** Technical report. [Paper](https://arxiv.org/abs/2610.15542), [model card](https://huggingface.co/swiss-ai/Apertus-v1.5-8B)

[04] ggml-org. **llama.cpp.** The engine that converts, quantizes and runs GGUF models, and that Ollama builds on. [Code](https://github.com/ggml-org/llama.cpp)

---
### 7.8 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Question 01: Expectations**

05.09 measures `qwen3.5:0.8b`, `qwen3.5:4b` and `lfm2.5` on the same question. Which of the three do you expect to give the best answer, and which to be the fastest? Note your guess and your reason.

(answer here)

**Question 02: How big is a model?**

Report two predictions of Section 7.2.1 that were off, with your numbers, and the reason for each gap. How large would a model with 7 billion parameters be in `Q8_0`, and would it fit into the memory of the Pi?

(answer here)

**Question 03: Parameters are not everything**

When would you choose `granite4:1b-h`, a 1.6 GB model built for long texts, over one of the larger models? What would you give up?

(answer here)

**Question 04: Total and active parameters**

`lfm2.5` uses 4 of its 32 experts for each token. Which of the two numbers, all parameters or the active ones, decides how much memory it needs, and which how fast it can generate? Note your guess; 05.09 tests it.

(answer here)

**Question 05: Open in a strong sense**

What makes Apertus open in a stronger sense than the other models on the list? We converted the three versions ourselves (Section 7.4.1). Which of the steps could change how a model behaves without anyone noticing it in the answers, and how would you check that a converted model behaves like the original?

(answer here)

**Question 06: Swiss languages**

Report the speed of each model in Section 7.4.2 and compare it with the file sizes in the table of Section 7.4. Which model answered best in Zurich German and in Romansh, rated separately for language and for truth? Did the amount of non-English training data show?

(answer here)

**Question 07: A model that sees**

What did `qwen3.5:4b` say about the Pi and about your own photo, and was it right? What did the larger image cost in prompt tokens and time?

(answer here)

**Question 08: Local models in your organisation**

Pick one responsibility from Section 7.3 and one idea from Section 7.5, and describe a concrete situation in your organisation where each would matter.

(answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)